In [1]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"


In [3]:
# ============================================================
# AudioLLM pipeline (disk-light via streaming LibriSpeech)
# - Audio -> EnCodec discrete tokens
# - Train ONLY a mapper into frozen LLM embedding space
# - Frozen LLM decodes text (toy ASR)
# - Uses streaming=True so it doesn't download big datasets
# ============================================================

# (Optional, helpful for debugging if anything goes wrong)
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

# Keep HF caches contained (models still download; dataset won't)
CACHE_DIR = "/content/hf_cache"
os.makedirs(CACHE_DIR, exist_ok=True)
os.environ["HF_HOME"] = CACHE_DIR
os.environ["TRANSFORMERS_CACHE"] = os.path.join(CACHE_DIR, "transformers")
os.environ["HF_DATASETS_CACHE"] = os.path.join(CACHE_DIR, "datasets")

!pip -q install "transformers>=4.49.0" datasets jiwer torchaudio accelerate

import re
import time
import math
import random
import itertools
from typing import List, Tuple

import torch
import torch.nn as nn
from torch.optim import AdamW
import torchaudio

from datasets import load_dataset
from jiwer import wer

from transformers import (
    AutoProcessor,
    EncodecModel,
    AutoTokenizer,
    AutoModelForCausalLM,
)

# -------------------------
# Setup
# -------------------------
SEED = 0
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# -------------------------
# Load frozen EnCodec + frozen LLM
# -------------------------
ENCODEC_ID = "facebook/encodec_24khz"
codec = EncodecModel.from_pretrained(ENCODEC_ID, cache_dir=CACHE_DIR).to(device).eval()
codec_processor = AutoProcessor.from_pretrained(ENCODEC_ID, cache_dir=CACHE_DIR)
for p in codec.parameters():
    p.requires_grad = False

LLM_ID = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(LLM_ID, cache_dir=CACHE_DIR)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# If you ever hit SDPA-related weirdness, you can force eager attention:
# llm = AutoModelForCausalLM.from_pretrained(LLM_ID, cache_dir=CACHE_DIR, attn_implementation="eager").to(device).eval()
llm = AutoModelForCausalLM.from_pretrained(LLM_ID, cache_dir=CACHE_DIR).to(device).eval()
for p in llm.parameters():
    p.requires_grad = False

llm_emb = llm.get_input_embeddings()
LLM_DIM = llm.config.n_embd
MAX_POS = int(getattr(llm.config, "n_positions", getattr(llm.config, "max_position_embeddings", 1024)))
print("LLM_DIM:", LLM_DIM, "| MAX_POS:", MAX_POS)

PROMPT_TEXT = "transcription: "
PROMPT_IDS = tokenizer(PROMPT_TEXT, add_special_tokens=False, return_tensors="pt").input_ids.squeeze(0)

# -------------------------
# Text normalization
# -------------------------
def normalize_text(s: str) -> str:
    s = s.lower()
    s = re.sub(r"[^a-z' ]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

# -------------------------
# Audio helpers: resample to 24kHz + crop duration
# -------------------------
TARGET_SR = int(codec_processor.sampling_rate)

def to_24khz_mono_np(audio_array, sr: int, max_audio_sec: float) -> torch.Tensor:
    """
    Returns torch float32 tensor shape (T,) at 24kHz, cropped to max_audio_sec.
    """
    x = torch.tensor(audio_array, dtype=torch.float32)

    # Some datasets provide shape (T,) already; if stereo appears, average to mono
    if x.dim() == 2:
        x = x.mean(dim=0)

    if sr != TARGET_SR:
        x = torchaudio.functional.resample(x, orig_freq=sr, new_freq=TARGET_SR)

    max_len = int(max_audio_sec * TARGET_SR)
    if x.numel() > max_len:
        x = x[:max_len]

    return x

# -------------------------
# EnCodec codes -> flat token ids
# -------------------------
def _to_codes_b_nq_t(audio_codes: torch.LongTensor) -> torch.LongTensor:
    # Common: (B, n_q, T) or (B, nb_chunks, n_q, Tchunk)
    if audio_codes.dim() == 4:
        B, nb, nq, Tc = audio_codes.shape
        return audio_codes.permute(0, 2, 1, 3).reshape(B, nq, nb * Tc)
    if audio_codes.dim() == 3:
        B, d1, d2 = audio_codes.shape
        if d1 <= 64:
            return audio_codes
        return audio_codes.unsqueeze(1)
    if audio_codes.dim() == 2:
        return audio_codes.unsqueeze(1)
    raise ValueError(f"Unexpected audio_codes shape: {tuple(audio_codes.shape)}")

def codes_to_flat_token_ids(audio_codes: torch.LongTensor, codebook_size: int) -> torch.LongTensor:
    codes = _to_codes_b_nq_t(audio_codes)  # (B, n_q, T)
    B, n_q, T = codes.shape
    offsets = (torch.arange(n_q, device=codes.device).view(1, n_q, 1) * codebook_size)
    flat = (codes + offsets).permute(0, 2, 1).reshape(B, T * n_q)
    return flat

@torch.no_grad()
def encode_audio_to_tokens(audio_tensor_24k: torch.Tensor, bandwidth: float) -> torch.LongTensor:
    """
    audio_tensor_24k: torch tensor (T,) at 24kHz
    returns: CPU LongTensor (L,)
    """
    # codec_processor expects numpy or list; use numpy for clarity
    inp = codec_processor(
        raw_audio=audio_tensor_24k.cpu().numpy(),
        sampling_rate=TARGET_SR,
        return_tensors="pt",
    )
    input_values = inp["input_values"].to(device)
    padding_mask = inp.get("padding_mask", None)
    if padding_mask is not None:
        padding_mask = padding_mask.to(device)

    enc = codec.encode(input_values, padding_mask=padding_mask, bandwidth=bandwidth)
    flat = codes_to_flat_token_ids(enc.audio_codes, codebook_size=codec.config.codebook_size)
    return flat.squeeze(0).detach().cpu()

# -------------------------
# Trainable mapper (ONLY trained part)
# -------------------------
class AudioTokenMapper(nn.Module):
    def __init__(self, codebook_size: int, max_codebooks: int, audio_emb_dim: int, llm_dim: int):
        super().__init__()
        self.audio_vocab = max_codebooks * codebook_size + 1
        self.pad_id = self.audio_vocab - 1
        self.embed = nn.Embedding(self.audio_vocab, audio_emb_dim, padding_idx=self.pad_id)
        self.proj = nn.Sequential(nn.LayerNorm(audio_emb_dim), nn.Linear(audio_emb_dim, llm_dim))

    def forward(self, token_ids: torch.LongTensor) -> torch.Tensor:
        return self.proj(self.embed(token_ids))

MAX_CODEBOOKS = int(getattr(codec.config, "num_codebooks", 32))
audio_mapper = AudioTokenMapper(
    codebook_size=codec.config.codebook_size,
    max_codebooks=MAX_CODEBOOKS,
    audio_emb_dim=256,
    llm_dim=LLM_DIM,
).to(device)

# -------------------------
# Streaming LibriSpeech (disk-light)
# -------------------------
# Splits exist like train.100 / train.360 / validation / test for clean. (We use train.100 + test)
train_stream = load_dataset("openslr/librispeech_asr", "clean", split="train.100", streaming=True)
test_stream  = load_dataset("openslr/librispeech_asr", "clean", split="test", streaming=True)

# Optional shuffle with small buffer (still streaming, doesn't download whole dataset)
train_stream = train_stream.shuffle(seed=SEED, buffer_size=512)

# Choose "not tiny" but still manageable
N_TRAIN = 400   # increase/decrease freely
N_TEST  = 50

# -------------------------
# Build batch safely: ensure total length <= MAX_POS
# -------------------------
def make_batch(audio_token_seqs: List[torch.LongTensor], texts: List[str]) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    embeds_list, attn_list, labels_list = [], [], []

    for a_tokens_cpu, txt in zip(audio_token_seqs, texts):
        tgt = normalize_text(txt)
        tgt_ids = tokenizer(tgt + tokenizer.eos_token, add_special_tokens=False, return_tensors="pt").input_ids.squeeze(0)

        prompt_len = PROMPT_IDS.numel()
        tgt_len = tgt_ids.numel()

        # Truncate transcript if it's too long to ever fit (rare)
        if prompt_len + tgt_len >= MAX_POS:
            keep = max(1, MAX_POS - prompt_len - 1)
            tgt_ids = tgt_ids[:keep]
            tgt_len = tgt_ids.numel()

        # Truncate audio tokens so that audio + prompt + transcript fit
        max_audio_len = MAX_POS - (prompt_len + tgt_len)
        max_audio_len = max(1, max_audio_len)
        a_tokens = a_tokens_cpu[:max_audio_len].to(device)

        # Mapper -> audio embeds
        a_embeds = audio_mapper(a_tokens.unsqueeze(0))  # (1, La, D)

        # Prompt + transcript text embeds
        prompt_ids = PROMPT_IDS.to(device)
        all_text_ids = torch.cat([prompt_ids, tgt_ids.to(device)], dim=0)
        t_embeds = llm_emb(all_text_ids.unsqueeze(0))

        full_embeds = torch.cat([a_embeds, t_embeds], dim=1).squeeze(0)

        # Labels: only supervise transcript tokens (ignore audio + prompt)
        La = a_embeds.size(1)
        Lp = prompt_ids.numel()
        labels = torch.full((La + Lp + tgt_ids.numel(),), -100, device=device, dtype=torch.long)
        labels[La + Lp : La + Lp + tgt_ids.numel()] = tgt_ids.to(device)

        attn = torch.ones((full_embeds.size(0),), device=device, dtype=torch.long)

        # Safety checks (catch issues before CUDA asserts)
        assert full_embeds.size(0) <= MAX_POS, (full_embeds.size(0), MAX_POS)
        assert int(a_tokens.max()) < audio_mapper.embed.num_embeddings, (int(a_tokens.max()), audio_mapper.embed.num_embeddings)
        ok = labels[labels != -100]
        if ok.numel() > 0:
            assert int(ok.max()) < llm.config.vocab_size, (int(ok.max()), llm.config.vocab_size)

        embeds_list.append(full_embeds)
        labels_list.append(labels)
        attn_list.append(attn)

    B = len(embeds_list)
    Lmax = max(x.size(0) for x in embeds_list)
    D = embeds_list[0].size(1)

    inputs_embeds = torch.zeros((B, Lmax, D), device=device)
    attention_mask = torch.zeros((B, Lmax), device=device, dtype=torch.long)
    labels = torch.full((B, Lmax), -100, device=device, dtype=torch.long)

    for i, (e, a, lab) in enumerate(zip(embeds_list, attn_list, labels_list)):
        L = e.size(0)
        inputs_embeds[i, :L, :] = e
        attention_mask[i, :L] = a
        labels[i, :L] = lab

    return inputs_embeds, attention_mask, labels

# -------------------------
# Training loop (streaming)
# -------------------------
BANDWIDTH = 3.0        # lower -> fewer tokens -> lower latency, lower quality
MAX_AUDIO_SEC = 2.0    # crop audio -> fewer tokens + fits context
BATCH_SIZE = 1
MAX_STEPS = 200        # use <= N_TRAIN if you want

optimizer = AdamW(audio_mapper.parameters(), lr=2e-4)

def stream_batches(ds_stream, batch_size: int, n_total: int):
    it = iter(ds_stream)
    produced = 0
    while produced < n_total:
        batch = []
        for _ in range(batch_size):
            if produced >= n_total:
                break
            ex = next(it)
            batch.append(ex)
            produced += 1
        if batch:
            yield batch

audio_mapper.train()
loss_ema = None
step = 0

for batch in stream_batches(train_stream, BATCH_SIZE, N_TRAIN):
    step += 1
    if step > MAX_STEPS:
        break

    texts = [ex["text"] for ex in batch]

    # Convert audio -> 24kHz -> discrete tokens
    t0 = time.time()
    token_seqs = []
    for ex in batch:
        arr = ex["audio"]["array"]
        sr = ex["audio"]["sampling_rate"]
        audio_24k = to_24khz_mono_np(arr, sr, max_audio_sec=MAX_AUDIO_SEC)
        token_seqs.append(encode_audio_to_tokens(audio_24k, bandwidth=BANDWIDTH))
    enc_time = time.time() - t0

    inputs_embeds, attention_mask, labels = make_batch(token_seqs, texts)

    out = llm(inputs_embeds=inputs_embeds, attention_mask=attention_mask, labels=labels)
    loss = out.loss

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(audio_mapper.parameters(), 1.0)
    optimizer.step()

    lv = float(loss.detach().cpu())
    loss_ema = lv if loss_ema is None else 0.9 * loss_ema + 0.1 * lv

    if step == 1 or step % 20 == 0:
        avg_tokens = sum(t.numel() for t in token_seqs) / len(token_seqs)
        print(f"step {step:04d} | loss {lv:.4f} | ema {loss_ema:.4f} | avg audio tokens {avg_tokens:.1f} | enc {enc_time:.2f}s")

audio_mapper.eval()
print("Training done.")

# -------------------------
# Inference: greedy decoding
# -------------------------
@torch.no_grad()
def transcribe(ex, bandwidth: float, max_audio_sec: float, max_new_tokens: int = 64) -> str:
    arr = ex["audio"]["array"]
    sr = ex["audio"]["sampling_rate"]
    audio_24k = to_24khz_mono_np(arr, sr, max_audio_sec=max_audio_sec)
    tokens = encode_audio_to_tokens(audio_24k, bandwidth=bandwidth)

    # Ensure room for prompt + generation
    room = MAX_POS - PROMPT_IDS.numel() - max_new_tokens
    room = max(1, room)
    tokens = tokens[:room].to(device)

    a_embeds = audio_mapper(tokens.unsqueeze(0))
    prompt_ids = PROMPT_IDS.to(device).unsqueeze(0)
    prefix = torch.cat([a_embeds, llm_emb(prompt_ids)], dim=1)

    out = llm(inputs_embeds=prefix, use_cache=True)
    past = out.past_key_values

    next_id = torch.argmax(out.logits[:, -1, :], dim=-1)  # (1,)
    gen = [next_id.item()]

    for _ in range(max_new_tokens - 1):
        out = llm(input_ids=next_id.unsqueeze(0), past_key_values=past, use_cache=True)
        past = out.past_key_values
        next_id = torch.argmax(out.logits[:, -1, :], dim=-1)
        gen.append(next_id.item())
        if next_id.item() == tokenizer.eos_token_id:
            break

    return normalize_text(tokenizer.decode(gen, skip_special_tokens=True))

# -------------------------
# Evaluation on streamed test subset
# -------------------------
test_examples = list(itertools.islice(iter(test_stream), N_TEST))

refs, hyps = [], []
for ex in test_examples:
    refs.append(normalize_text(ex["text"]))
    hyps.append(transcribe(ex, bandwidth=BANDWIDTH, max_audio_sec=MAX_AUDIO_SEC, max_new_tokens=64))

print("\nSample predictions:")
for i in range(min(5, len(refs))):
    print("REF:", refs[i])
    print("HYP:", hyps[i])
    print("---")

print("WER:", wer(refs, hyps))

# -------------------------
# Trade-off sweep: bandwidth vs tokens/time (single example)
# -------------------------
@torch.no_grad()
def codec_tradeoff_stats(ex, bandwidth: float, max_audio_sec: float):
    arr = ex["audio"]["array"]
    sr = ex["audio"]["sampling_rate"]
    audio_24k = to_24khz_mono_np(arr, sr, max_audio_sec=max_audio_sec)

    t0 = time.time()
    tokens = encode_audio_to_tokens(audio_24k, bandwidth=bandwidth)
    enc_time = time.time() - t0

    dur_s = audio_24k.numel() / TARGET_SR
    tok_per_s = tokens.numel() / max(dur_s, 1e-6)

    return {
        "bandwidth_kbps": float(bandwidth),
        "duration_s": float(dur_s),
        "num_tokens": int(tokens.numel()),
        "tokens_per_second": float(tok_per_s),
        "encode_time_s": float(enc_time),
    }

example = test_examples[0]
bandwidths = getattr(codec.config, "target_bandwidths", [1.5, 3.0, 6.0, 12.0, 24.0])

print("\nTrade-off sweep (cropped audio):")
for bw in bandwidths:
    print(codec_tradeoff_stats(example, bw, MAX_AUDIO_SEC))


device: cuda


Loading weights:   0%|          | 0/252 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


LLM_DIM: 768 | MAX_POS: 1024


...

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

step 0001 | loss 6.2760 | ema 6.2760 | avg audio tokens 600.0 | enc 0.25s
step 0020 | loss 6.9644 | ema 6.2331 | avg audio tokens 600.0 | enc 0.04s
step 0040 | loss 5.9463 | ema 5.9048 | avg audio tokens 600.0 | enc 0.03s
step 0060 | loss 5.5776 | ema 5.8044 | avg audio tokens 600.0 | enc 0.04s
step 0080 | loss 5.3268 | ema 5.8126 | avg audio tokens 600.0 | enc 0.05s
step 0100 | loss 4.9854 | ema 5.4284 | avg audio tokens 600.0 | enc 0.03s
step 0120 | loss 5.2344 | ema 5.7288 | avg audio tokens 600.0 | enc 0.04s
step 0140 | loss 4.5595 | ema 5.3896 | avg audio tokens 600.0 | enc 0.04s
step 0160 | loss 4.5866 | ema 5.1987 | avg audio tokens 600.0 | enc 0.04s
step 0180 | loss 5.4814 | ema 5.5477 | avg audio tokens 600.0 | enc 0.04s
step 0200 | loss 5.2027 | ema 5.1141 | avg audio tokens 600.0 | enc 0.04s
Training done.

Sample predictions:
REF: concord returned to its place amidst the tents
HYP: the first of the three main reasons why we have to be able to see the world in the same way a

I formed the bridge between audio and text by **tokenizing the waveform with EnCodec into discrete audio IDs**, then **training a small mapping module (embedding + projection) that converts those audio-token IDs into DistilGPT-2’s 768-dim embedding space**, and finally **feeding the frozen LLM the sequence `[audio_embeds] + "transcription:"` so it can generate the transcript autoregressively**.

Because the **LLM is frozen**, I **only train the mapping module** using a language-model loss on the transcript tokens (I mask out loss on the audio prefix and prompt), so the mapper learns to produce an audio-conditioned prefix that the LLM can decode into text.

The **quality–latency–tokens trade-off** is visible in my bandwidth sweep on a 2-second clip: **1.5 kbps → 300 tokens, 3 kbps → 600 tokens, 6 kbps → 1200 tokens, 12 kbps → 2400 tokens, 24 kbps → 4800 tokens**, meaning **higher bandwidth gives a richer audio representation (potentially higher quality) but produces many more tokens that increase compute and delay**.

In my system, **latency is driven mainly by the number of tokens the LLM must process**, since transformer attention cost increases rapidly with sequence length, while EnCodec’s encode time changed only slightly (~0.03–0.04 s in my run).

A key constraint is that **DistilGPT-2 has a 1024-token context window**, so **high bandwidth or longer audio can exceed the context and force truncation**, which directly hurts transcription quality.

The parts that most strongly affect final output quality are **(1) the codec settings (bandwidth/token budget), (2) the mapping module’s capacity to compress and represent audio tokens in the LLM space, and (3) the LLM’s decoding behavior (e.g., repetition under greedy decoding)**—which matches my observed outputs (repetitive/hallucinated text and high WER).
